# Predicción de demanda con Machine Learning — Rossmann Store Sales

## Configuración A: solo variables de contexto (+ experimento con día de la semana)

### Objetivo del trabajo

Medir cuánto mejora un modelo de Machine Learning (**XGBoost**) al predecir las ventas diarias de cada tienda cuando le agregamos **variables temporales** (retardos de demanda y estacionalidad).

Para eso comparamos dos configuraciones:

| Configuración | Qué variables usa |
|---|---|
| **A** (este notebook) | Solo variables de **contexto**: promedio histórico de la tienda, promoción, feriados |
| **B** (sección final) | Contexto + variables **temporales** (lags y calendario) |

En este notebook resolvemos la **Configuración A**, que además sirve de punto de partida para B. Incluye:

- 2 **baselines** (modelos "tontos" contra los que hay que ganar).
- El modelo **XGBoost con Config A**.
- Un experimento extra: **Config A + día de la semana**.

### Cómo se evalúa (idea clave)

Predecimos **el futuro con datos del pasado**. Por eso **no** mezclamos filas al azar: usamos una validación temporal (*expanding window*) con 5 folds. Las métricas son **MAE, RMSE, MAPE y R²**.

### Cómo usar este notebook

1. Ejecutar las celdas **en orden**, de arriba hacia abajo.
2. En la sección 3 se pide subir el archivo `train.csv` del dataset de Rossmann (Kaggle: *Rossmann Store Sales*).

## 1. Importación de librerías

- `pandas` / `numpy`: manejo de datos y cálculos.
- `matplotlib` / `seaborn`: gráficos.
- `sklearn.metrics`: funciones para calcular el error del modelo.
- `XGBRegressor`: el modelo de Machine Learning (árboles de decisión con *gradient boosting*) para predecir un número (regresión).

In [ ]:
import io
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from google.colab import files
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from xgboost import XGBRegressor

sns.set_theme(style="whitegrid")
pd.set_option("display.width", 160)
pd.set_option("display.max_columns", 30)

## 2. Parámetros del experimento

Definimos todo lo "configurable" en un solo lugar, para que sea fácil de cambiar y de explicar.

- **`TARGET`**: lo que queremos predecir (`Sales`, ventas diarias de una tienda).
- **`FEATURES_A`**: variables de entrada de la Config A.
  - `StoreMean`: promedio histórico de ventas de la tienda (se calcula más adelante, **solo con datos de entrenamiento**).
  - `Promo`: si ese día la tienda tenía promoción (0/1).
  - `StateHoliday`: feriado estatal (`0` = no, `a`, `b`, `c` = tipos de feriado).
  - `SchoolHoliday`: si había feriado escolar (0/1).
- **`FEATURES_A_DOW`**: igual que A pero agregando `DayOfWeek` (1 = lunes … 7 = domingo). Es el experimento extra.
- **Validación temporal**: `N_FOLDS` = 5 pruebas, cada una evalúa sobre `TEST_DAYS` = 42 días (6 semanas). `GAP_DAYS` = días de separación entre train y test (0 = ninguno).
- **`XGB_PARAMS`**: hiperparámetros del modelo (se mantienen fijos para que la comparación A vs B sea justa).

In [ ]:
TARGET = "Sales"

FEATURES_A     = ["StoreMean", "Promo", "StateHoliday", "SchoolHoliday"]
FEATURES_A_DOW = FEATURES_A + ["DayOfWeek"]

# Validación temporal
N_FOLDS   = 5     # cantidad de pruebas
TEST_DAYS = 42    # días de cada bloque de test (6 semanas)
GAP_DAYS  = 0     # días de separación entre train y test

# Hiperparámetros de XGBoost (fijos en A y B para comparar de forma justa)
XGB_PARAMS = dict(
    n_estimators=300,        # cantidad de árboles
    learning_rate=0.05,      # cuánto aporta cada árbol nuevo
    max_depth=8,             # profundidad máxima de cada árbol
    subsample=0.8,           # fracción de filas usada por árbol
    colsample_bytree=0.8,    # fracción de columnas usada por árbol
    min_child_weight=5,      # mínimo de datos por hoja (evita sobreajuste)
    tree_method="hist",
    enable_categorical=True, # permite usar columnas categóricas (StateHoliday)
    random_state=42,         # semilla: mismos resultados en cada corrida
    n_jobs=-1,
)

## 3. Carga del dataset

Ejecutá la celda y seleccioná el archivo **`train.csv`** desde tu computadora cuando aparezca el botón *Elegir archivos*.

El archivo se lee directamente desde memoria (`io.BytesIO`), sin depender de Google Drive ni de rutas fijas. Así cualquier integrante del grupo puede correr el notebook subiendo su propia copia del dataset.

Columnas esperadas: `Store, DayOfWeek, Date, Sales, Customers, Open, Promo, StateHoliday, SchoolHoliday`.

In [ ]:
print("Seleccioná el archivo train.csv de Rossmann:")
uploaded = files.upload()

nombre_archivo = list(uploaded.keys())[0]
df_raw = pd.read_csv(io.BytesIO(uploaded[nombre_archivo]), parse_dates=["Date"], low_memory=False)

print(f"\nArchivo cargado: {nombre_archivo}")
print(f"Filas: {df_raw.shape[0]:,}  |  Columnas: {df_raw.shape[1]}")
df_raw.head()

## 4. Limpieza y preparación de los datos

Pasos (en este orden):

1. **`StateHoliday` a texto**: en el CSV mezcla `0` (número) y `'0'` (texto), y eso confunde al modelo. Lo pasamos todo a `str`.
2. **Filtrar `Open == 1` y `Sales > 0`**: un día con la tienda cerrada tiene ventas 0 por definición. Dejarlo ensuciaría el promedio de ventas y las métricas (sobre todo el MAPE, que divide por las ventas reales).
3. **Ordenar por fecha y tienda**: indispensable para una validación temporal.
4. **`StateHoliday` como `category`**: XGBoost la trata como categoría y no como número.

In [ ]:
def limpiar_datos(df_raw):
    """Devuelve el dataset listo para modelar (solo días con tienda abierta y ventas > 0)."""
    df = df_raw.copy()
    df["StateHoliday"] = df["StateHoliday"].astype(str)       # 1. unificar tipos
    df = df[(df["Open"] == 1) & (df[TARGET] > 0)]             # 2. solo días válidos
    df = df.sort_values(["Date", "Store"]).reset_index(drop=True)  # 3. orden temporal
    df["StateHoliday"] = df["StateHoliday"].astype("category")     # 4. categórica
    return df

df = limpiar_datos(df_raw)

print(f"Filas antes de limpiar : {len(df_raw):,}")
print(f"Filas después          : {len(df):,}")
print(f"Tiendas                : {df['Store'].nunique()}")
print(f"Rango de fechas        : {df['Date'].min().date()} → {df['Date'].max().date()}")
df.head()

## 5. Exploración rápida

Antes de modelar conviene mirar los datos. Dos preguntas simples:

- ¿Cambian las ventas según el **día de la semana**? Si sí, `DayOfWeek` debería ayudar al modelo (lo probamos en el experimento A + DOW).
- ¿Las **promociones** aumentan las ventas?

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

df.groupby("DayOfWeek")[TARGET].mean().plot(kind="bar", ax=axes[0], color="steelblue")
axes[0].set_title("Ventas promedio por día de la semana (1 = lunes)")
axes[0].set_xlabel("DayOfWeek"); axes[0].set_ylabel("Ventas promedio")

df.groupby("Promo")[TARGET].mean().plot(kind="bar", ax=axes[1], color="darkorange")
axes[1].set_title("Ventas promedio según promoción (0 = no, 1 = sí)")
axes[1].set_xlabel("Promo"); axes[1].set_ylabel("Ventas promedio")

plt.tight_layout()
plt.show()

## 6. Validación temporal (*expanding window*)

### ¿Por qué no un split aleatorio 80/20?

Si mezclamos filas al azar, el modelo entrena con días **posteriores** al que tiene que predecir, y ve los días vecinos: eso es **filtración del futuro** (*leakage*) y da un error falsamente bajo. En el uso real nunca conocemos el futuro.

### Cómo lo hacemos

Armamos **5 folds**. En cada uno:

- **Train** = todo lo anterior a una fecha de corte (la ventana *crece* en cada fold).
- **Test** = los 42 días siguientes.

```
Fold 1: [=========== train ===========][test]
Fold 2: [============= train ============][test]
...
Fold 5: [=========================== train ==========================][test]
```

La función trabaja sobre **fechas únicas** (no sobre filas), así todas las tiendas de un mismo día quedan del mismo lado.

In [ ]:
def make_folds(dates, n_folds=N_FOLDS, test_days=TEST_DAYS, gap=GAP_DAYS):
    """
    Genera los folds de validación temporal.
    Devuelve una lista de tuplas (fechas_train, fechas_test).
    """
    unique = np.sort(pd.Series(dates).unique())   # fechas únicas ordenadas
    folds = []
    for k in range(n_folds, 0, -1):               # el último fold termina al final del dataset
        test_end   = len(unique) - (k - 1) * test_days
        test_start = test_end - test_days
        train_end  = test_start - gap
        folds.append((unique[:train_end], unique[test_start:test_end]))
    return folds

folds = make_folds(df["Date"])

# Tabla resumen de los folds
resumen_folds = pd.DataFrame([{
    "Fold": i + 1,
    "Train desde": pd.Timestamp(tr[0]).date(), "Train hasta": pd.Timestamp(tr[-1]).date(),
    "Test desde":  pd.Timestamp(te[0]).date(), "Test hasta":  pd.Timestamp(te[-1]).date(),
    "Días train": len(tr), "Días test": len(te),
} for i, (tr, te) in enumerate(folds)])
resumen_folds

## 7. La variable `StoreMean` (promedio histórico por tienda)

Es la variable de contexto más importante: le dice al modelo "esta tienda vende, en promedio, X por día".

**Regla de oro: se calcula SOLO con el train de cada fold.** Si usáramos todo el dataset, el promedio incluiría ventas del período de test (el futuro) y estaríamos haciendo trampa.

Detalle: si una tienda aparece en test pero no en train, no tiene promedio propio; en ese caso usamos el **promedio global del train** (`fillna`).

In [ ]:
def add_store_mean(train, test):
    """Agrega StoreMean a train y test usando únicamente datos de train."""
    train, test = train.copy(), test.copy()
    medias = train.groupby("Store")[TARGET].mean()          # promedio por tienda (solo train)
    train["StoreMean"] = train["Store"].map(medias)
    test["StoreMean"]  = test["Store"].map(medias).fillna(train[TARGET].mean())  # tienda nueva -> media global
    return train, test

# Ejemplo con el primer fold
tr_dates, te_dates = folds[0]
_tr, _te = add_store_mean(df[df["Date"].isin(tr_dates)], df[df["Date"].isin(te_dates)])
_te[["Date", "Store", "Sales", "StoreMean"]].head()

## 8. Métricas de evaluación

Comparamos lo que el modelo predijo (`y_pred`) con lo que realmente pasó (`y_true`):

| Métrica | Qué mide | Cómo leerla |
|---|---|---|
| **MAE** | Error absoluto promedio | "En promedio me equivoco en X unidades de venta" |
| **RMSE** | Raíz del error cuadrático medio | Como el MAE, pero **castiga más los errores grandes** |
| **MAPE (%)** | Error porcentual promedio | "Me equivoco X % respecto al valor real" |
| **R²** | Proporción de variación explicada | 1 = perfecto, 0 = no mejora a predecir el promedio |

In [ ]:
def calcular_metricas(y_true, y_pred):
    """Devuelve MAE, RMSE, MAPE (%) y R² de una predicción."""
    mae  = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    mape = np.mean(np.abs((y_true - y_pred) / y_true)) * 100   # válido porque Sales > 0
    r2   = r2_score(y_true, y_pred)
    return {"MAE": mae, "RMSE": rmse, "MAPE": mape, "R2": r2}

## 9. Baselines (modelos de referencia)

Un modelo sofisticado solo vale la pena si le **gana a una regla simple**. Usamos dos:

1. **Media por tienda**: predice siempre el promedio histórico de esa tienda (ignora promos, feriados y días).
2. **Media por tienda × día de la semana**: predice el promedio de esa tienda *para ese día de la semana* (ej.: "los sábados esta tienda vende 4.400").

Ambos se calculan solo con datos de train y se aplican sobre test, sin ver el futuro.

In [ ]:
def baseline_media_tienda(train, test):
    medias = train.groupby("Store")[TARGET].mean()
    return test["Store"].map(medias).fillna(train[TARGET].mean()).values


def baseline_media_tienda_dow(train, test):
    medias = train.groupby(["Store", "DayOfWeek"])[TARGET].mean().rename("pred")
    pred = test.join(medias, on=["Store", "DayOfWeek"])["pred"]
    return pred.fillna(train[TARGET].mean()).values

## 10. Entrenamiento y evaluación (funciones)

Para **cada fold**:

1. Separamos train y test por fechas.
2. Calculamos `StoreMean` solo con el train.
3. Entrenamos con **`fit(X_train, y_train)`**: el modelo aprende la relación entre variables y ventas.
4. Predecimos con **`predict(X_test)`**: **no** le pasamos las ventas reales del test; así sabemos si realmente funciona.
5. Comparamos la predicción con la realidad usando las métricas.

`correr_baseline` hace lo mismo pero sin entrenar: aplica la regla simple.

In [ ]:
def _fila_resultado(name, i, tr, te, y_true, y_pred, segundos):
    """Arma una fila con las métricas de un fold."""
    fila = {"modelo": name, "fold": i + 1,
            "train_desde": tr["Date"].min().date(), "train_hasta": tr["Date"].max().date(),
            "test_desde":  te["Date"].min().date(), "test_hasta":  te["Date"].max().date(),
            "n_train": len(tr), "n_test": len(te), "segundos": round(segundos, 1)}
    fila.update(calcular_metricas(y_true, y_pred))
    return fila


def correr_xgboost(df, features, name):
    """Evalúa XGBoost con las variables `features` en todos los folds."""
    filas = []
    for i, (tr_dates, te_dates) in enumerate(make_folds(df["Date"])):
        t0 = time.time()
        tr, te = df[df["Date"].isin(tr_dates)], df[df["Date"].isin(te_dates)]
        tr, te = add_store_mean(tr, te)                      # sin filtración del futuro

        model = XGBRegressor(**XGB_PARAMS)
        model.fit(tr[features], tr[TARGET])                  # aprende
        pred = model.predict(te[features])                   # predice SIN ver las ventas reales

        fila = _fila_resultado(name, i, tr, te, te[TARGET].values, pred, time.time() - t0)
        filas.append(fila)
        print(f"[{name}] fold {i+1}/{N_FOLDS}  MAE={fila['MAE']:.1f}  MAPE={fila['MAPE']:.2f}%  R2={fila['R2']:.4f}")
    return pd.DataFrame(filas)


def correr_baseline(df, funcion, name):
    """Evalúa un baseline (regla simple) en todos los folds."""
    filas = []
    for i, (tr_dates, te_dates) in enumerate(make_folds(df["Date"])):
        t0 = time.time()
        tr, te = df[df["Date"].isin(tr_dates)], df[df["Date"].isin(te_dates)]
        pred = funcion(tr, te)
        fila = _fila_resultado(name, i, tr, te, te[TARGET].values, pred, time.time() - t0)
        filas.append(fila)
        print(f"[{name}] fold {i+1}/{N_FOLDS}  MAE={fila['MAE']:.1f}  MAPE={fila['MAPE']:.2f}%  R2={fila['R2']:.4f}")
    return pd.DataFrame(filas)

## 11. Ejecución de los experimentos

Corremos los 4 modelos sobre los mismos folds:

1. Baseline: media por tienda
2. Baseline: media por tienda × día de la semana
3. **XGBoost Config A** (solo contexto)
4. **XGBoost Config A + DayOfWeek**

> ⏱️ Puede tardar unos minutos, sobre todo los dos XGBoost.

In [ ]:
res_b1 = correr_baseline(df, baseline_media_tienda,     "Baseline: media por tienda")
res_b2 = correr_baseline(df, baseline_media_tienda_dow, "Baseline: media tienda x dia")
res_a  = correr_xgboost(df, FEATURES_A,     "XGBoost Config A")
res_adow = correr_xgboost(df, FEATURES_A_DOW, "XGBoost Config A + DOW")

por_fold = pd.concat([res_b1, res_b2, res_a, res_adow], ignore_index=True)

## 12. Resultados

### 12.1 Resumen: promedio y desvío entre folds

Reportamos el **promedio** de cada métrica sobre los 5 folds y su **desvío estándar** (`std`): un desvío chico significa que el modelo rinde parecido en distintos períodos, es decir, que es estable.

In [ ]:
metricas = ["MAE", "RMSE", "MAPE", "R2"]
orden = list(por_fold["modelo"].unique())

resumen = por_fold.groupby("modelo")[metricas].agg(["mean", "std"]).loc[orden]
resumen.columns = [f"{m}_{s}" for m, s in resumen.columns]
resumen.round(4)

### 12.2 Comparación visual

Cuanto **más bajo** MAE y MAPE, mejor. Cuanto **más alto** R², mejor.

In [ ]:
medias = por_fold.groupby("modelo")[metricas].mean().loc[orden]

fig, axes = plt.subplots(1, 3, figsize=(16, 4.5))
for ax, m, titulo in zip(axes, ["MAE", "MAPE", "R2"], ["MAE (menor = mejor)", "MAPE % (menor = mejor)", "R² (mayor = mejor)"]):
    sns.barplot(x=medias[m].values, y=medias.index, ax=ax, color="steelblue")
    ax.set_title(titulo); ax.set_xlabel(m); ax.set_ylabel("")
    for p in ax.patches:
        ax.annotate(f"{p.get_width():.2f}" if m != "MAE" else f"{p.get_width():.0f}",
                    (p.get_width(), p.get_y() + p.get_height() / 2), va="center", ha="left", fontsize=9)
plt.tight_layout()
plt.show()

### 12.3 Métricas fold por fold

Sirve para ver si hay algún período especialmente difícil (por ejemplo, uno con temporada alta).

In [ ]:
por_fold[["modelo", "fold", "test_desde", "test_hasta", "MAE", "RMSE", "MAPE", "R2"]].round(4)

### 12.4 Mejora respecto de los baselines

Cuánto reduce el error XGBoost Config A frente a cada regla simple, en términos de MAE.

In [ ]:
mae = medias["MAE"]
a = mae["XGBoost Config A"]
for ref in ["Baseline: media por tienda", "Baseline: media tienda x dia"]:
    print(f"Config A vs {ref}: MAE {mae[ref]:.1f} -> {a:.1f}  ({(1 - a / mae[ref]) * 100:+.1f}% de reducción del error)")

adow = mae["XGBoost Config A + DOW"]
print(f"\nAgregar DayOfWeek a la Config A: MAE {a:.1f} -> {adow:.1f}  ({(1 - adow / a) * 100:+.1f}% de reducción)")

## 13. Guardar resultados

Guardamos las tablas para usarlas después en la comparación con la **Configuración B** y en el informe. Se descargan a tu computadora.

In [ ]:
por_fold.to_csv("metricas_por_fold_A.csv", index=False)
resumen.round(4).to_csv("metricas_resumen_A.csv")

files.download("metricas_por_fold_A.csv")
files.download("metricas_resumen_A.csv")

## 14. Conclusiones de la Configuración A

*(Completar con los números obtenidos al correr el notebook.)*

- **Baselines**: la media por tienda logra un MAE de ___ y un R² de ___; sumar el día de la semana lo baja a ___.
- **Config A**: XGBoost con solo variables de contexto obtiene MAE ___ / MAPE ___ % / R² ___. ¿Le gana a los baselines? ¿Por cuánto?
- **Experimento A + DOW**: agregar el día de la semana mejora el MAE en ___ %, lo que indica que ___.
- **Estabilidad**: el desvío del R² entre folds es ___, es decir que el modelo es ___ a lo largo del tiempo.
- **Limitación**: la Config A no conoce lo que pasó los días anteriores (no tiene *lags*). Eso es exactamente lo que pone a prueba la Configuración B.

---
# Configuración B (retardos de demanda y patrones completos de estacionalidad)

En esta sección armamos la **Configuración B**: todo lo de la A **más variables temporales**. La idea es medir cuánto mejora el modelo cuando además de saber "cómo es la tienda" (contexto) sabe "qué pasó los días anteriores" (lags) y "en qué momento del año estamos" (calendario).

### Qué es un *lag* (retardo)

Un lag es **lo que vendió esa misma tienda hace N días**:

| Variable | Significado |
|---|---|
| `lag_1`  | ventas de ayer |
| `lag_7`  | ventas de hace una semana (mismo día de la semana) |
| `lag_14` | ventas de hace dos semanas |
| `lag_28` | ventas de hace cuatro semanas |

Cada lag apunta a **una sola fila**: la de la fecha exacta `hoy − N días`. No se suman ni se promedian días.

### Dos decisiones de diseño (importantes para el informe)

1. **Los lags se calculan por fecha de calendario, no por posición de fila.** Los días con la tienda cerrada fueron eliminados en la limpieza. Si tomáramos "la fila anterior", el `lag_1` de un lunes traería el sábado y no el domingo. Buscando por fecha exacta, si ese día no existe el lag queda vacío (`NaN`), y XGBoost sabe trabajar con valores vacíos.
2. **Horizonte de predicción = 1 día hacia adelante.** Para predecir el día D usamos lo que ya pasó hasta D−1. Por eso un lag de un día *dentro del período de test* es válido: ese día ya ocurrió cuando hacemos la predicción. Los lags **nunca miran al futuro**, así que no hay filtración (*leakage*).

## 15. Creación de los lags (por fecha)

El truco con `pandas`:

1. Tomamos una tabla con `Store`, `Date` y `Sales`.
2. Le **corremos la fecha N días hacia adelante**: las ventas del 13 pasan a figurar el 20 (si N = 7).
3. Hacemos un `merge` (unión) con la tabla original por `Store` + `Date`.

Así cada fila "recibe" lo que vendió la misma tienda N días antes. Si ese día no existe, queda `NaN`.

In [ ]:
LAGS = [1, 7, 14, 28]

def agregar_lags(df, lags=LAGS):
    """Agrega columnas lag_N = ventas de la misma tienda hace N días (por fecha exacta)."""
    df = df.copy()
    base = df[["Store", "Date", TARGET]]
    for n in lags:
        corrida = base.copy()
        corrida["Date"] = corrida["Date"] + pd.Timedelta(days=n)       # las ventas de hoy "aparecen" en hoy+N
        corrida = corrida.rename(columns={TARGET: f"lag_{n}"})
        df = df.merge(corrida, on=["Store", "Date"], how="left")        # si no existe ese día -> NaN
    return df

df_b = agregar_lags(df)
df_b[["Store", "Date", "DayOfWeek", "Sales", "lag_1", "lag_7", "lag_14", "lag_28"]].head(10)

### Verificación a mano

Siempre conviene comprobar con un caso concreto que el lag hace lo que creemos. Tomamos una tienda y un día cualquiera y buscamos a mano la venta de 7 días antes.

In [ ]:
tienda = df_b["Store"].iloc[0]
fila = df_b[(df_b["Store"] == tienda)].iloc[100]
fecha_ref = fila["Date"] - pd.Timedelta(days=7)
real = df_b[(df_b["Store"] == tienda) & (df_b["Date"] == fecha_ref)][TARGET]

print(f"Tienda {tienda}, fecha {fila['Date'].date()}")
print(f"lag_7 calculado     : {fila['lag_7']}")
print(f"Venta del {fecha_ref.date()}   : {real.iloc[0] if len(real) else 'no existe (NaN)'}")
print(f"\nPorcentaje de NaN por lag:")
print((df_b[[f'lag_{n}' for n in LAGS]].isna().mean() * 100).round(2).astype(str) + " %")

## 16. Variables de calendario (estacionalidad)

Además de los lags, le damos al modelo **en qué momento del año** estamos, para que aprenda patrones repetitivos (más ventas en diciembre, picos a fin de mes, etc.):

- `DayOfWeek`: día de la semana (ya existía).
- `Month`: mes (1–12).
- `WeekOfYear`: semana del año (1–52).
- `DayOfMonth`: día del mes (1–31).

In [ ]:
def agregar_calendario(df):
    df = df.copy()
    df["Month"]      = df["Date"].dt.month
    df["WeekOfYear"] = df["Date"].dt.isocalendar().week.astype(int)
    df["DayOfMonth"] = df["Date"].dt.day
    return df

df_b = agregar_calendario(df_b)

LAG_COLS      = [f"lag_{n}" for n in LAGS]
CALENDAR_COLS = ["DayOfWeek", "Month", "WeekOfYear", "DayOfMonth"]

FEATURES_A_CAL = FEATURES_A + CALENDAR_COLS               # A + calendario
FEATURES_A_LAG = FEATURES_A + LAG_COLS                    # A + lags
FEATURES_B     = FEATURES_A + CALENDAR_COLS + LAG_COLS    # Config B completa
print("Config B usa:", FEATURES_B)

## 17. ¿Por qué los lags sirven? (gráfico)

Si las ventas de hace una semana se parecen a las de hoy, la nube de puntos `lag_7` vs `Sales` tiene que quedar pegada a una diagonal.

In [ ]:
muestra = df_b.dropna(subset=["lag_7"]).sample(min(20000, len(df_b)), random_state=42)

fig, ax = plt.subplots(figsize=(6, 5))
ax.scatter(muestra["lag_7"], muestra[TARGET], s=4, alpha=0.25, color="steelblue")
lim = muestra[[TARGET, "lag_7"]].max().max()
ax.plot([0, lim], [0, lim], color="red", lw=1, label="y = x (ventas iguales)")
ax.set_xlabel("Ventas hace 7 días (lag_7)"); ax.set_ylabel("Ventas de hoy")
ax.set_title("Relación entre lag_7 y las ventas de hoy"); ax.legend()
plt.show()

print(f"Correlación lag_7 vs Sales: {muestra['lag_7'].corr(muestra[TARGET]):.3f}")

## 18. Experimentos de la Configuración B (con ablación)

Una **ablación** consiste en sumar las piezas de a una para ver cuál aporta la mejora. Corremos sobre los mismos folds:

| Experimento | Variables |
|---|---|
| Config A | contexto (ya corrido arriba) |
| A + calendario | contexto + día de la semana, mes, semana, día del mes |
| A + lags | contexto + lag_1, 7, 14, 28 |
| **Config B** | contexto + calendario + lags |

> ⏱️ Son 3 modelos XGBoost × 5 folds.

In [ ]:
res_acal = correr_xgboost(df_b, FEATURES_A_CAL, "XGBoost A + calendario")
res_alag = correr_xgboost(df_b, FEATURES_A_LAG, "XGBoost A + lags")
res_b    = correr_xgboost(df_b, FEATURES_B,     "XGBoost Config B")

por_fold_total = pd.concat([por_fold, res_acal, res_alag, res_b], ignore_index=True)

## 19. Resultados comparativos A vs B

### 19.1 Tabla resumen (promedio y desvío entre folds)

In [ ]:
orden_total = list(por_fold_total["modelo"].unique())
resumen_total = por_fold_total.groupby("modelo")[metricas].agg(["mean", "std"]).loc[orden_total]
resumen_total.columns = [f"{m}_{s}" for m, s in resumen_total.columns]
resumen_total.round(4)

### 19.2 Ablación: ¿qué aporta cada pieza?

Barras de MAE (menor = mejor) y R² (mayor = mejor) para todos los experimentos.

In [ ]:
medias_total = por_fold_total.groupby("modelo")[metricas].mean().loc[orden_total]

fig, axes = plt.subplots(1, 2, figsize=(15, 4.5))
for ax, m, t in zip(axes, ["MAE", "R2"], ["MAE (menor = mejor)", "R² (mayor = mejor)"]):
    sns.barplot(x=medias_total[m].values, y=medias_total.index, ax=ax, color="steelblue")
    ax.set_title(t); ax.set_xlabel(m); ax.set_ylabel("")
    for p in ax.patches:
        ax.annotate(f"{p.get_width():.0f}" if m == "MAE" else f"{p.get_width():.3f}",
                    (p.get_width(), p.get_y() + p.get_height() / 2), va="center", ha="left", fontsize=9)
plt.tight_layout()
plt.show()

### 19.3 MAE por fold: A vs B

Si B gana en **todos** los folds, la mejora es estable y no depende de un período puntual.

In [ ]:
comp = por_fold_total[por_fold_total["modelo"].isin(["XGBoost Config A", "XGBoost Config B"])]
fig, ax = plt.subplots(figsize=(8, 4))
sns.barplot(data=comp, x="fold", y="MAE", hue="modelo", ax=ax)
ax.set_title("MAE por fold: Config A vs Config B"); ax.set_xlabel("Fold")
plt.show()

mae_t = medias_total["MAE"]
print(f"MAE promedio: A = {mae_t['XGBoost Config A']:.1f}  ->  B = {mae_t['XGBoost Config B']:.1f}")
print(f"Reducción del error de B respecto de A: {(1 - mae_t['XGBoost Config B'] / mae_t['XGBoost Config A']) * 100:.1f} %")

## 20. Análisis del último fold

Entrenamos de nuevo A y B sobre el **último fold** (el más reciente) para guardar las predicciones y poder graficarlas.

In [ ]:
def predecir_fold(df, features, idx=-1):
    """Entrena en el fold `idx` y devuelve (modelo, test con columna 'pred')."""
    tr_dates, te_dates = make_folds(df["Date"])[idx]
    tr, te = add_store_mean(df[df["Date"].isin(tr_dates)], df[df["Date"].isin(te_dates)])
    model = XGBRegressor(**XGB_PARAMS).fit(tr[features], tr[TARGET])
    te = te.assign(pred=model.predict(te[features]))
    return model, te

_, te_a = predecir_fold(df_b, FEATURES_A)
model_b, te_b = predecir_fold(df_b, FEATURES_B)

### 20.1 Ventas reales vs predichas en el tiempo

Sumamos las ventas de todas las tiendas por día y comparamos con lo que predijo cada configuración. Es el gráfico que más se entiende: se ve si el modelo "sigue" los picos y valles reales.

In [ ]:
diario = pd.DataFrame({
    "Real":     te_b.groupby("Date")[TARGET].sum(),
    "Config A": te_a.groupby("Date")["pred"].sum(),
    "Config B": te_b.groupby("Date")["pred"].sum(),
})

fig, ax = plt.subplots(figsize=(12, 4.5))
ax.plot(diario.index, diario["Real"],     color="black",      lw=2,   label="Real")
ax.plot(diario.index, diario["Config A"], color="darkorange", lw=1.5, label="Predicción Config A")
ax.plot(diario.index, diario["Config B"], color="steelblue",  lw=1.5, label="Predicción Config B")
ax.set_title("Ventas totales por día en el último fold (42 días de test)")
ax.set_ylabel("Ventas totales"); ax.legend()
plt.show()

### 20.2 Importancia de variables en Config B

Muestra qué variables usó más el modelo para decidir. Si `lag_7` o `StoreMean` pesan mucho, confirma que el pasado de la tienda es lo que más ayuda.

In [ ]:
imp = pd.Series(model_b.feature_importances_, index=FEATURES_B).sort_values()
fig, ax = plt.subplots(figsize=(7, 5))
imp.plot(kind="barh", ax=ax, color="steelblue")
ax.set_title("Importancia de variables — XGBoost Config B"); ax.set_xlabel("Importancia")
plt.show()

### 20.3 Error por día de la semana

¿En qué días falla más cada configuración? (MAE en el último fold.)

In [ ]:
err = pd.DataFrame({
    "Config A": te_a.assign(e=(te_a[TARGET] - te_a["pred"]).abs()).groupby("DayOfWeek")["e"].mean(),
    "Config B": te_b.assign(e=(te_b[TARGET] - te_b["pred"]).abs()).groupby("DayOfWeek")["e"].mean(),
})
err.plot(kind="bar", figsize=(8, 4), color=["darkorange", "steelblue"])
plt.title("MAE por día de la semana (1 = lunes)"); plt.ylabel("MAE"); plt.xlabel("DayOfWeek")
plt.show()

## 21. Guardar resultados de la comparación

In [ ]:
por_fold_total.to_csv("metricas_por_fold_A_y_B.csv", index=False)
resumen_total.round(4).to_csv("metricas_resumen_A_y_B.csv")
files.download("metricas_por_fold_A_y_B.csv")
files.download("metricas_resumen_A_y_B.csv")

## 22. Conclusiones de la Configuración B

*(Completar con los números obtenidos.)*

- **A vs B**: el MAE pasa de ___ a ___ (___ % de reducción) y el R² de ___ a ___.
- **Ablación**: el calendario aporta ___ y los lags aportan ___. La pieza que más mejora es ___.
- **Estabilidad**: B gana en ___ de los 5 folds.
- **Variables más importantes** (gráfico 20.2): ___.
- **Limitaciones**: los lags requieren conocer las ventas de los días anteriores (horizonte de 1 día); si hubiera que predecir 42 días de una vez, `lag_1` y `lag_7` no estarían disponibles.
- **Conclusión general**: la ingeniería de variables temporales ___ el desempeño del modelo respecto de usar solo contexto.